# DDOS Identification - example usage

## Preprocessing

### Load dataset

In [1]:
import pandas as pd
import numpy as np

# Loading logic
df = pd.read_csv("/kaggle/input/sliit-final/data/data_for_demo.csv")

# Uncomment to view the first 5 rows
# print(df.head(5))

# Done
print("File loaded")

df.shape

File loaded


(127946, 84)

### IP address segmenting 

In [2]:
# Convert IP strings into octets efficiently using NumPy
def ip_to_octets(ip_series, prefix):
    octets = np.stack(ip_series.str.split('.').apply(lambda x: list(map(int, x))))
    return pd.DataFrame(octets, columns=[f"{prefix} Octet{i+1}" for i in range(4)])

# Apply the optimized function
src_octets = ip_to_octets(df["Src IP"], "Src IP")
dst_octets = ip_to_octets(df["Dst IP"], "Dst IP")

# Concatenate with original DataFrame
df = pd.concat([df, src_octets, dst_octets], axis=1).drop(columns=["Src IP", "Dst IP"])

# Delete unused vars
del src_octets
del dst_octets

# Uncomement below to view the first 5 rows
print(df.head(5))

# Done
print("IP segmentation Done")

                                  Flow ID  Src Port  Dst Port  Protocol  \
0  172.31.69.25-18.219.211.138-80-34408-6     34408        80         6   
1  172.31.69.25-18.219.211.138-80-40676-6     40676        80         6   
2    192.168.2.109-203.73.24.75-4020-80-6      4020        80         6   
3  172.31.69.25-18.219.211.138-80-36102-6     36102        80         6   
4  172.31.69.25-18.219.211.138-80-44284-6     44284        80         6   

                Timestamp  Flow Duration  Tot Fwd Pkts  Tot Bwd Pkts  \
0  15/02/2018 07:24:59 PM        5032233             3             5   
1  15/02/2018 07:26:24 PM       10993127             3             5   
2  12/06/2010 03:12:15 PM         479425             3             7   
3  15/02/2018 07:08:53 PM        9042123             1             1   
4  15/02/2018 07:19:56 PM        1907579             1             1   

   TotLen Fwd Pkts  TotLen Bwd Pkts  ...   Idle Min  Label  Src IP Octet1  \
0            603.0            972.0  ..

### Drop unneeded columns

In [3]:
dropping_cols = ["Flow ID"]
for col in dropping_cols:
    if col in df.columns:
        df.drop(columns=col, inplace=True)

numerical_cols = df.select_dtypes(include=np.number).columns.tolist()
non_numerical_cols = df.select_dtypes(exclude=np.number).columns.tolist()

### Load preprocessing models

In [4]:
import joblib
imputer = joblib.load("/kaggle/input/sliit-final/model/imputer.joblib")
scaler = joblib.load("/kaggle/input/sliit-final/model/scaler.joblib")

### Handle infinite values

In [5]:
for col in numerical_cols:
    print(f"{col}: {df[col].isin([np.inf, -np.inf]).sum()}")

Src Port: 0
Dst Port: 0
Protocol: 0
Flow Duration: 0
Tot Fwd Pkts: 0
Tot Bwd Pkts: 0
TotLen Fwd Pkts: 0
TotLen Bwd Pkts: 0
Fwd Pkt Len Max: 0
Fwd Pkt Len Min: 0
Fwd Pkt Len Mean: 0
Fwd Pkt Len Std: 0
Bwd Pkt Len Max: 0
Bwd Pkt Len Min: 0
Bwd Pkt Len Mean: 0
Bwd Pkt Len Std: 0
Flow Byts/s: 176
Flow Pkts/s: 490
Flow IAT Mean: 0
Flow IAT Std: 0
Flow IAT Max: 0
Flow IAT Min: 0
Fwd IAT Tot: 0
Fwd IAT Mean: 0
Fwd IAT Std: 0
Fwd IAT Max: 0
Fwd IAT Min: 0
Bwd IAT Tot: 0
Bwd IAT Mean: 0
Bwd IAT Std: 0
Bwd IAT Max: 0
Bwd IAT Min: 0
Fwd PSH Flags: 0
Bwd PSH Flags: 0
Fwd URG Flags: 0
Bwd URG Flags: 0
Fwd Header Len: 0
Bwd Header Len: 0
Fwd Pkts/s: 0
Bwd Pkts/s: 0
Pkt Len Min: 0
Pkt Len Max: 0
Pkt Len Mean: 0
Pkt Len Std: 0
Pkt Len Var: 0
FIN Flag Cnt: 0
SYN Flag Cnt: 0
RST Flag Cnt: 0
PSH Flag Cnt: 0
ACK Flag Cnt: 0
URG Flag Cnt: 0
CWE Flag Count: 0
ECE Flag Cnt: 0
Down/Up Ratio: 0
Pkt Size Avg: 0
Fwd Seg Size Avg: 0
Bwd Seg Size Avg: 0
Fwd Byts/b Avg: 0
Fwd Pkts/b Avg: 0
Fwd Blk Rate Avg: 0
Bwd B

In [6]:
df.replace([np.inf, -np.inf], np.NaN, inplace=True)
print("Replacing completed")

Replacing completed


### Impute missing values

In [7]:
df[numerical_cols] = pd.DataFrame(imputer.transform(df[numerical_cols]), columns=numerical_cols, index=df.index)  # DO NOT remove the index=xxx attribute, else NaN happens
for col in numerical_cols:
    print(f"{col}: {df[col].isin([np.inf, -np.inf, np.NaN]).sum()}")
# Uncomement below to view the first 5 rows
# df.head(5)

# done
print("Imputing done")

Src Port: 0
Dst Port: 0
Protocol: 0
Flow Duration: 0
Tot Fwd Pkts: 0
Tot Bwd Pkts: 0
TotLen Fwd Pkts: 0
TotLen Bwd Pkts: 0
Fwd Pkt Len Max: 0
Fwd Pkt Len Min: 0
Fwd Pkt Len Mean: 0
Fwd Pkt Len Std: 0
Bwd Pkt Len Max: 0
Bwd Pkt Len Min: 0
Bwd Pkt Len Mean: 0
Bwd Pkt Len Std: 0
Flow Byts/s: 0
Flow Pkts/s: 0
Flow IAT Mean: 0
Flow IAT Std: 0
Flow IAT Max: 0
Flow IAT Min: 0
Fwd IAT Tot: 0
Fwd IAT Mean: 0
Fwd IAT Std: 0
Fwd IAT Max: 0
Fwd IAT Min: 0
Bwd IAT Tot: 0
Bwd IAT Mean: 0
Bwd IAT Std: 0
Bwd IAT Max: 0
Bwd IAT Min: 0
Fwd PSH Flags: 0
Bwd PSH Flags: 0
Fwd URG Flags: 0
Bwd URG Flags: 0
Fwd Header Len: 0
Bwd Header Len: 0
Fwd Pkts/s: 0
Bwd Pkts/s: 0
Pkt Len Min: 0
Pkt Len Max: 0
Pkt Len Mean: 0
Pkt Len Std: 0
Pkt Len Var: 0
FIN Flag Cnt: 0
SYN Flag Cnt: 0
RST Flag Cnt: 0
PSH Flag Cnt: 0
ACK Flag Cnt: 0
URG Flag Cnt: 0
CWE Flag Count: 0
ECE Flag Cnt: 0
Down/Up Ratio: 0
Pkt Size Avg: 0
Fwd Seg Size Avg: 0
Bwd Seg Size Avg: 0
Fwd Byts/b Avg: 0
Fwd Pkts/b Avg: 0
Fwd Blk Rate Avg: 0
Bwd Byts/

### Scale dataset

In [8]:
df[numerical_cols] = pd.DataFrame(scaler.transform(df[numerical_cols]), columns=numerical_cols, index=df.index)

# Uncomement below to view the first 5 rows
# df.head(5)

# done
print("Scaling done")

Scaling done


### Mapping Label (if present)

In [9]:
if 'Label' in df.columns:
    X = df.drop(columns=['Label'])
    y = df['Label']
    y = y.map({"Benign": 0, "ddos": 1})
else:
    X = df
    y = None

**End of preprocessing**

## Model loading

### Class definitions

In [10]:
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader

# Autoencoder Class
class Autoencoder(nn.Module):
    def __init__(self, input_dim): # x -> x' -> x'' 
        super(Autoencoder, self).__init__()
        # Encoder
        self.encoder = nn.Sequential(
            nn.Linear(input_dim, 64),
            nn.SELU(),
            nn.Linear(64, 32),
            nn.SELU(),
            nn.Linear(32, 8),
            nn.SELU()
        )
        # Decoder
        self.decoder = nn.Sequential(
            nn.Linear(8, 32),
            nn.SELU(),
            nn.Linear(32, 64),
            nn.SELU(),
            nn.Linear(64, input_dim),
            nn.SELU()
        )
        
    def forward(self, x):
        encoded = self.encoder(x)
        decoded = self.decoder(encoded)
        return decoded

# LSTM Class
class LSTMModel(nn.Module):
    def __init__(self, input_size, hidden_size=128, num_layers=2, output_size=2):
        super(LSTMModel, self).__init__()
        self.hidden_size = hidden_size
        self.num_layers = num_layers
        
        # LSTM layer
        self.lstm = nn.LSTM(input_size, hidden_size, num_layers, batch_first=True)
        
        # Fully connected layer
        self.fc = nn.Linear(hidden_size, output_size)

    def forward(self, x):
        h0 = torch.zeros(self.num_layers, x.size(0), self.hidden_size).to(x.device)
        c0 = torch.zeros(self.num_layers, x.size(0), self.hidden_size).to(x.device)

        out, _ = self.lstm(x, (h0, c0))  # LSTM forward pass
        out = self.fc(out[:, -1, :])  # Take the output of the last time step
        return out

In [11]:
import json
import torch
import joblib
from sklearn.linear_model import LogisticRegression

# Model files
log_reg_file = "/kaggle/input/sliit-final/model/log_reg.pkl"
autoencoder_file = "/kaggle/input/sliit-final/model/autoencoder.pt"
autoencoder_params_file = "*.pth" # state_dict(); i.e. model
autoencoder_hyperparams_file = "*.json" # 
autoencoder_errors_file = "*.json"
lstm_file = "/kaggle/input/sliit-final/model/lstm.pt"
lstm_params_file = "*.pth"
lstm_hyperparams_file = "*.json"

# Loading model hyperparams
# with open(autoencoder_hyperparams_file, "r") as f:
#     autoencoder_hyperparams = json.load(f)
# with open(lstm_hyperparams_file, "r") as f:
#     lstm_hyperparams = json.load(f)
# with open(autoencoder_errors_file, "r") as f:
#     autoencoder_errors = json.load(f)
    

# Initialize model with loaded hyperparams
# autoencoder = Autoencoder(**autoencoder_hyperparams)
# lstm = LSTM(**lstm_hyperparams)

# # Load model weights
# autoencoder.load_state_dict(torch.load(autoencoder_params_file))
# lstm.load_state_dict(torch.load(lstm_params_file))

# # Set torch models to evaluate mode (i.e. predictions mode)
# autoencoder.eval()
# lstm.eval()

# Load autoencoder
autoencoder = torch.load(autoencoder_file)
autoencoder.eval()

# Load LSTM
lstm = torch.load(lstm_file)
lstm.eval()

# Load logistic regression
log_reg = joblib.load(log_reg_file)

print("Loading complete")

Loading complete


<ipython-input-11-dad743276ec2>:38: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  autoencoder = torch.load(autoencoder_file)
<ipython-input-11-dad743276ec2>:42: FutureWarnin

## Predictions pipeline

### Data Definitions

In [12]:
numerical_cols = X.select_dtypes(include=np.number).columns.tolist()
non_numerical_cols = X.select_dtypes(exclude=np.number).columns.tolist()
X_numpy = df.values
X_tensor = torch.FloatTensor(X[numerical_cols].values)

### Logistic Regression Predictions

In [13]:
# Get predictions
preds_log_reg = log_reg.predict(X[numerical_cols])

# Get indices of predicted DDoS and predicted Benign
logistic_ddos_indices = np.where(preds_log_reg == 1)[0]
logistic_benign_indices = np.where(preds_log_reg == 0)[0]

# Get indices of actual DDoS
ddos_indices = np.where(y == 1)[0]
benign_indices = np.where(y == 0)[0]

# print
print(f"predicted ddos count: {logistic_ddos_indices.shape}")
print(f"actual ddos count: {ddos_indices.shape}")

# X.iloc[logistic_ddos_indices].tail()
# matches (i.e. intersection)
log_ddos_intersection = set(ddos_indices) & set(logistic_ddos_indices)
print(f"ddos matches: {len(log_ddos_intersection)}")

preds_log_reg_df = X.iloc[logistic_ddos_indices]
preds_log_reg_df.to_csv("log_reg_results.csv")

predicted ddos count: (64898,)
actual ddos count: (64746,)
ddos matches: 64395


### Autoencoder predictions

In [14]:
# Set to evaluate mode
autoencoder.eval()
with torch.no_grad():
    reconstructed = autoencoder(X_tensor)  # Autoencoder output
    reconstruction_error = torch.mean((X_tensor - reconstructed) ** 2, dim=1)  # Compute MSE per sample

# threshold = autoencoder_errors.threshold
threshold = 0.017251603054865443
autoencoder_preds = (reconstruction_error > threshold).cpu().numpy().astype(int)  # Convert to binary

autoencoder_ddos_indices = np.where(autoencoder_preds == 1)[0]  # Get indices of DDoS
autoencoder_benign_indices = np.where(autoencoder_preds == 0)[0] 
print(autoencoder_ddos_indices.shape)
print(autoencoder_benign_indices.shape)

# Get indices of actual DDoS
ddos_indices = np.where(y == 1)[0]
benign_indices = np.where(y == 0)[0]

autoencoder_ddos_intersection = set(ddos_indices) & set(autoencoder_ddos_indices)
print(f"ddos matches: {len(autoencoder_ddos_intersection)}")

preds_log_reg_df = X.iloc[autoencoder_ddos_indices]
preds_log_reg_df.to_csv("autoencoder_results.csv")

(15478,)
(112468,)
ddos matches: 12643


### Combine Layer 1 suspected

In [15]:
log_autoencoder_ddos_intersection = log_ddos_intersection & autoencoder_ddos_intersection
log_autoencoder_ddos_union = log_ddos_intersection | autoencoder_ddos_intersection
autoencoder_specific_ddos_matches = log_autoencoder_ddos_union - log_ddos_intersection
log_specific_ddos_matches = log_autoencoder_ddos_union - autoencoder_ddos_intersection

print(len(log_specific_ddos_matches))
print(len(autoencoder_specific_ddos_matches))

51841
89


In [16]:
# ddos_indices_combined = np.union1d(logistic_ddos_indices, autoencoder_ddos_indices)  # Merge indices
# X_ddos = X.iloc[ddos_indices_combined]  # Extract DDoS samples
# X_ddos_tensor = torch.tensor(X_ddos[numerical_cols]).to(device)  # Convert for LSTM

# Define device (use GPU if available, otherwise CPU)
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

ddos_indices_combined = np.union1d(logistic_ddos_indices, autoencoder_ddos_indices)  # Merge indices
X_sus = X.iloc[ddos_indices_combined]  # Extract DDoS samples
X_sus_numpy = X_sus[numerical_cols].values  # Convert DataFrame to numpy array
X_ddos_tensor = torch.tensor(X_sus_numpy, dtype=torch.float32).to(device)  # Convert to tensor for LSTM
X_ddos_tensor = X_ddos_tensor.unsqueeze(1)  # Shape: (batch_size, 1, input_size)

### LSTM predictions (i.e. Layer 2 predictions)

In [17]:
lstm.eval()
with torch.no_grad():
    lstm_outputs = lstm(X_ddos_tensor)
    _, lstm_preds = torch.max(lstm_outputs, 1)  # Convert to class predictions

lstm_ddos_indices = ddos_indices_combined[np.where(lstm_preds.cpu().numpy() == 1)[0]]  # Get final DDoS indices
print(len(lstm_ddos_indices))

lstm_ddos_intersection = set(ddos_indices) & set(lstm_ddos_indices)
print(f"ddos matches: {len(lstm_ddos_intersection)}")

preds_lstm_df = X.iloc[lstm_ddos_indices]
preds_lstm_df.to_csv("lstm_results.csv")

61570
ddos matches: 61308


In [18]:
X_ddos_tensor.shape

torch.Size([67703, 1, 87])

### Get additional info

In [19]:
# Get additional details from the original dataset
original_dataset = pd.read_csv("/kaggle/input/sliit-final/data/data_for_demo.csv")
ddos_data_final = original_dataset.iloc[lstm_ddos_indices]  # Extract corresponding rows

# Select key fields like source IP, destination IP, and ports
key_fields = [ 
    'Src IP',
    'Src Port',
    'Dst IP',
    'Dst Port',
    'Timestamp'
]
ddos_key_info = ddos_data_final[key_fields]

# Get first 10
ddos_key_info.head(10)

# Store
ddos_key_info.to_csv("ddos_key_info.csv")